# Prepare a 20,000-image Re-LAION-Art pool

This notebook uses the LAION-Art Hub page's current recommendation, `datasets.load_dataset("laion/relaion-art", split="train", streaming=True)`, then passes a locally saved selection to the official [`img2dataset` workflow](https://github.com/rom1504/img2dataset/blob/main/dataset_examples/laion-art.md). Authenticate separately with `hf auth login`; no token is stored here. It does **not** create captions or concept groups.

AMP says that its images are resized to 1024×1024, but neither the paper nor repository specifies an interpolation or crop policy for constructing this pool. This notebook therefore uses a documented deterministic policy: resize directly to 1024×1024 with Pillow LANCZOS (no crop). An existing, valid 1024×1024 PNG is copied byte-for-byte instead of being re-encoded.

Install notebook-only dependencies in the active environment if needed:

```bash
uv pip install datasets img2dataset pyarrow pandas pillow
```


In [ ]:
from concurrent.futures import ProcessPoolExecutor
import json
import os
from pathlib import Path
import shutil
import subprocess

from datasets import load_dataset
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image, ImageOps

# Configuration
N = 20_000
RANDOM_SEED = 2025
STREAM_SHUFFLE_BUFFER = 10_000
HF_DATASET_ID = "laion/relaion-art"
METADATA_CACHE = Path("dataset/laion_art/cache")
TEMP_DOWNLOAD_ROOT = Path("dataset/laion_art/tmp")
FINAL_IMAGE_DIR = Path("dataset/laion_art/clean")
FINAL_METADATA_CSV = Path("dataset/laion_art/metadata.csv")
CPU_WORKERS = max(1, (os.cpu_count() or 2) - 1)
DOWNLOAD_PROCESSES = min(16, CPU_WORKERS)
DOWNLOAD_THREADS = 32
TARGET_SIZE = (1024, 1024)
AMP_ID_COLUMN = "amp_sample_id"

# Selection-specific paths prevent incremental downloads from mixing configurations.
RUN_DIR = TEMP_DOWNLOAD_ROOT / f"n{N}_seed{RANDOM_SEED}_buffer{STREAM_SHUFFLE_BUFFER}"
SELECTED_PARQUET = RUN_DIR / "selected.parquet"
SELECTION_CONFIG = RUN_DIR / "selection.json"
RAW_IMAGE_DIR = RUN_DIR / "downloads"

for directory in (METADATA_CACHE, RUN_DIR, FINAL_IMAGE_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
selection_config = {
    "dataset": HF_DATASET_ID,
    "split": "train",
    "streaming": True,
    "count": N,
    "seed": RANDOM_SEED,
    "shuffle_buffer": STREAM_SHUFFLE_BUFFER,
}


def selection_is_reusable(parquet_path, config_path, expected_config):
    if not parquet_path.is_file() or not config_path.is_file():
        return False
    try:
        saved_config = json.loads(config_path.read_text())
        row_count = pq.ParquetFile(parquet_path).metadata.num_rows
    except (OSError, ValueError, json.JSONDecodeError):
        return False
    return saved_config == expected_config and row_count == expected_config["count"]


def select_streaming_records(dataset_id, count, seed, buffer_size, cache_dir):
    """Shuffle a bounded stream reproducibly and consume only the requested records."""
    stream = load_dataset(
        dataset_id,
        split="train",
        streaming=True,
        cache_dir=str(cache_dir),
    )
    stream = stream.shuffle(seed=seed, buffer_size=buffer_size)
    records = []
    for selection_index, record in enumerate(stream.take(count)):
        record = dict(record)  # Preserve every field supplied by the Hub dataset.
        if AMP_ID_COLUMN in record:
            raise KeyError(f"Reserved column already exists: {AMP_ID_COLUMN}")
        record[AMP_ID_COLUMN] = f"{selection_index:012d}"
        records.append(record)
    if len(records) != count:
        raise ValueError(f"Requested {count:,} records, but the stream yielded {len(records):,}.")
    return records


if not selection_is_reusable(SELECTED_PARQUET, SELECTION_CONFIG, selection_config):
    selected_records = select_streaming_records(
        HF_DATASET_ID,
        N,
        RANDOM_SEED,
        STREAM_SHUFFLE_BUFFER,
        METADATA_CACHE,
    )
    temporary_parquet = SELECTED_PARQUET.with_suffix(".parquet.part")
    pq.write_table(pa.Table.from_pylist(selected_records), temporary_parquet, compression="zstd")
    temporary_parquet.replace(SELECTED_PARQUET)
    temporary_config = SELECTION_CONFIG.with_suffix(".json.part")
    temporary_config.write_text(json.dumps(selection_config, indent=2) + "\n")
    temporary_config.replace(SELECTION_CONFIG)

selected_count = pq.ParquetFile(SELECTED_PARQUET).metadata.num_rows
assert selected_count == N
print(f"Saved deterministic selection of {selected_count:,} streamed records to {SELECTED_PARQUET}")


In [ ]:
# Detect the Hub dataset's URL/caption casing and retain our stable ID in sidecars.
# Incremental mode skips shards completed by an earlier interrupted run.
selected_columns = set(pq.read_schema(SELECTED_PARQUET).names)
URL_COLUMN = next((name for name in ("URL", "url") if name in selected_columns), None)
CAPTION_COLUMN = next((name for name in ("TEXT", "text", "caption") if name in selected_columns), None)
if URL_COLUMN is None:
    raise KeyError(f"No URL column found in streamed metadata: {sorted(selected_columns)}")
caption_args = ["--caption_col", CAPTION_COLUMN] if CAPTION_COLUMN else []

command = [
    "img2dataset",
    "--url_list", str(SELECTED_PARQUET),
    "--input_format", "parquet",
    "--url_col", URL_COLUMN,
    *caption_args,
    "--output_format", "files",
    "--output_folder", str(RAW_IMAGE_DIR),
    "--resize_mode", "no",  # Preserve downloaded bytes for controlled final processing.
    "--processes_count", str(DOWNLOAD_PROCESSES),
    "--thread_count", str(DOWNLOAD_THREADS),
    "--number_sample_per_shard", "1000",
    "--save_additional_columns", json.dumps([AMP_ID_COLUMN]),
    "--incremental_mode", "incremental",
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)


In [ ]:
def valid_final_png(path):
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            return image.format == "PNG" and image.size == TARGET_SIZE
    except (OSError, ValueError):
        return False


def find_downloads(raw_directory):
    """Map amp_sample_id to downloaded file using img2dataset's JSON sidecars."""
    found = {}
    for sidecar in raw_directory.rglob("*.json"):
        try:
            payload = json.loads(sidecar.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        amp_sample_id = payload.get(AMP_ID_COLUMN)
        if amp_sample_id is None:
            continue
        candidates = [
            path for path in sidecar.parent.glob(sidecar.stem + ".*")
            if path.suffix.lower() not in {".json", ".txt"}
        ]
        if candidates:
            found[str(amp_sample_id)] = candidates[0]
    return found


def process_image(task):
    """Validate, normalize to the required PNG/size, and delete raw only on success."""
    amp_sample_id, raw_path_string, final_path_string = task
    raw_path = Path(raw_path_string) if raw_path_string else None
    final_path = Path(final_path_string)

    if valid_final_png(final_path):
        if raw_path and raw_path.exists() and raw_path != final_path:
            raw_path.unlink()
        return amp_sample_id, "complete", "", str(final_path)
    if raw_path is None or not raw_path.is_file():
        return amp_sample_id, "download_failed", "No downloaded image was recorded", ""

    temporary = final_path.with_suffix(".png.part")
    try:
        with Image.open(raw_path) as image:
            image.verify()
        with Image.open(raw_path) as image:
            original_format = image.format
            original_size = image.size
            if original_format == "PNG" and original_size == TARGET_SIZE:
                shutil.copyfile(raw_path, temporary)
            else:
                image = ImageOps.exif_transpose(image).convert("RGB")
                if image.size != TARGET_SIZE:
                    image = image.resize(TARGET_SIZE, resample=Image.Resampling.LANCZOS)
                image.save(temporary, format="PNG", optimize=False)
        if not valid_final_png(temporary):
            raise ValueError("Final PNG validation failed")
        temporary.replace(final_path)
        raw_path.unlink()
        return amp_sample_id, "complete", "", str(final_path)
    except Exception as error:
        temporary.unlink(missing_ok=True)
        return amp_sample_id, "decode_failed", f"{type(error).__name__}: {error}", ""


In [ ]:
# Only metadata for the selected 20,000 rows is held in memory; image pixels never are.
metadata = pq.read_table(SELECTED_PARQUET).to_pandas()
metadata[AMP_ID_COLUMN] = metadata[AMP_ID_COLUMN].astype(str)
downloaded = find_downloads(RAW_IMAGE_DIR)
tasks = [
    (
        amp_sample_id,
        str(downloaded[amp_sample_id]) if amp_sample_id in downloaded else "",
        str(FINAL_IMAGE_DIR / f"{amp_sample_id}.png"),
    )
    for amp_sample_id in metadata[AMP_ID_COLUMN]
]

with ProcessPoolExecutor(max_workers=CPU_WORKERS) as executor:
    outcomes = list(executor.map(process_image, tasks, chunksize=16))

status = pd.DataFrame(outcomes, columns=[AMP_ID_COLUMN, "status", "error", "final_path"])
metadata = metadata.merge(status, on=AMP_ID_COLUMN, how="left", validate="one_to_one")
metadata.to_csv(FINAL_METADATA_CSV, index=False)

print(metadata["status"].value_counts(dropna=False))
print("Metadata:", FINAL_METADATA_CSV)
metadata.head()


In [ ]:
# Final invariant check. Failed rows remain in metadata.csv with their error/status.
completed = metadata.loc[metadata["status"] == "complete", "final_path"]
invalid = [path for path in completed if not valid_final_png(Path(path))]
assert not invalid, f"Invalid final files: {invalid[:5]}"
print(f"Validated {len(completed):,} final 1024×1024 PNG images")
